# 107 — DS5 Trajectory Visualizations (Exploratory)

**Purpose:** Exploratory notebook to assess and draft four visualization types proposed for the
thesis presentation. All figures use real DS5 data (German adolescents, RCT, N=573 behavioral).

| Version | Description |
|---|---|
| A | Average vs Individual Trajectories — thin individual lines + thick group mean, colored by condition |
| B | Same Average, Different Dynamics — 3 real users with similar x_mean but different behavioral patterns |
| C | Intervention Response Phenotypes — classify participants by response type (strong/weak/non-responder/increase) |
| D | Decision-Level Intervention Flow — outcome distribution by condition + per-participant scatter |

**Datasets used:**
- `outputs/cleaned/ds5_behavioral_clean.csv` — event-level (288k rows)
- `outputs/features/ds5/ds5_behavior_features.csv` — user-level behavioral features
- `outputs/model_outputs/phenotyping/ds5_phenotype_feature_matrix.csv` — model-derived features incl. sig_hat
- `outputs/model_outputs/phenotyping/ds5_behav_only_cluster_assignments.csv` — Path A clusters (C1/C2/C3)

**Status:** Standalone exploratory — no `outputs/` companion file required.

---
## 0 — Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

ROOT     = Path.cwd().parent
CLEANED  = ROOT / 'outputs' / 'cleaned'
FEATURES = ROOT / 'outputs' / 'features' / 'ds5'
PHENO    = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
print('Loading behavioral data (~288k rows)...')
df_beh = pd.read_csv(CLEANED / 'ds5_behavioral_clean.csv', low_memory=False)

print('Loading feature files...')
df_feat       = pd.read_csv(FEATURES / 'ds5_behavior_features.csv')
df_pheno_feat = pd.read_csv(PHENO / 'ds5_phenotype_feature_matrix.csv')
df_clusters   = pd.read_csv(PHENO / 'ds5_behav_only_cluster_assignments.csv')

print(f'Behavioral rows: {len(df_beh):,}')
print(f'Behavioral participants: {df_beh["participant_id_canonical"].nunique()}')
print(f'Feature file participants: {len(df_feat)}')
print(f'Phenotype matrix participants: {len(df_pheno_feat)}')
print(f'Cluster assignment participants: {len(df_clusters)}')

In [ ]:
# Map study_condition from features file onto behavioral data
condition_map = df_feat.set_index('participant_id_canonical')['study_condition'].to_dict()
df_beh['study_condition'] = df_beh['participant_id_canonical'].map(condition_map)

# study_condition is used as a groupby key below, and pandas drops NaN keys silently.
# Expected: exactly 1 participant ('test-<participant-id>', a test
# account with no pre-survey record — see memory ds5_structural_discoveries.md §3), so
# the effective N for every Version A/C figure is 572, not the 573 printed above.
_unmapped = sorted(df_beh.loc[df_beh['study_condition'].isna(), 'participant_id_canonical'].unique())
print(f'Participants without study_condition (dropped by later groupbys): {len(_unmapped)} -> {_unmapped}')
assert len(_unmapped) <= 1, 'Unexpected number of participants without a study_condition'

# Parse local timestamp and derive hour-of-day
df_beh['event_timestamp_local'] = pd.to_datetime(df_beh['event_timestamp_local'], errors='coerce')
df_beh['event_hour'] = df_beh['event_timestamp_local'].dt.hour

# Integer day index — floor to nearest day; keep as float so NaN rows drop silently in groupby
df_beh['day_int'] = np.floor(pd.to_numeric(df_beh['days_since_first_event'], errors='coerce'))

n_nan = df_beh['day_int'].isna().sum()
if n_nan > 0:
    print(f'Note: {n_nan} rows have NaN day_int and will be excluded from daily aggregations')

# Summary
cond_counts = df_beh.groupby('study_condition')['participant_id_canonical'].nunique()
print('Participants with behavioral data per condition:')
print(cond_counts)
valid_days = df_beh['day_int'].dropna()
print(f'\nObservation window: day {int(valid_days.min())} to day {int(valid_days.max())}')

---
## Version A — Average vs Individual Trajectories

**Message:** Even when the group average shows a moderate intervention effect, individual trajectories diverge substantially.
This motivates phenotyping rather than relying on average effects alone.

**Left panel:** Overall opening rate (all events, all conditions comparable).  
**Right panel:** Dismissal rate, all three conditions shown (control saw no breathing delay but could still dismiss — mean rate_dismissed = 0.179).

**Y-axis note:** `P(open)` = proportion of events where the user opened the app. For the control group,
no friction was shown, so their opening rate reflects unimpeded use (no breathing delay was added - the user could instantly select open or dismiss).

In [ ]:
# Per-user per-day aggregation
daily = (
    df_beh.groupby(['participant_id_canonical', 'day_int', 'study_condition'])
    .agg(
        n_events   = ('event_outcome_simple', 'count'),
        n_opened   = ('event_outcome_simple', lambda x: (x == 'opened').sum()),
        n_dismissed= ('event_outcome_simple', lambda x: (x == 'dismissed').sum()),
        n_closed   = ('event_outcome_simple', lambda x: (x == 'closed').sum()),
    )
    .reset_index()
)
daily['open_rate']    = daily['n_opened']    / daily['n_events']
daily['dismiss_rate'] = daily['n_dismissed'] / daily['n_events']

# Filter to participants with >= 14 active days
active_days = daily.groupby('participant_id_canonical')['day_int'].count()
eligible    = set(active_days[active_days >= 14].index)
print(f'Participants with >= 14 active days: {len(eligible)} / {daily["participant_id_canonical"].nunique()}')

daily_elig = daily[daily['participant_id_canonical'].isin(eligible)].copy()
pid_condition = (
    daily_elig.drop_duplicates('participant_id_canonical')
    .set_index('participant_id_canonical')['study_condition']
    .to_dict()
)

In [ ]:
# Build participant x day pivot tables, then smooth with 7-day rolling average
def make_pivot_smooth(daily_df, value_col, window=7):
    pivot = daily_df.pivot_table(
        index='participant_id_canonical', columns='day_int', values=value_col
    )
    # Rolling along day axis: transpose so days are the index, apply rolling, transpose back
    return pivot.T.rolling(window=window, min_periods=1).mean().T

pivot_open    = make_pivot_smooth(daily_elig, 'open_rate')
pivot_dismiss = make_pivot_smooth(daily_elig, 'dismiss_rate')

print(f'Pivot shape: {pivot_open.shape}  (participants x days)')

In [ ]:
COLORS = {'control': "#218D0E", 'default': '#1976D2', 'reintervention': '#F57C00'}
LABELS = {'control': 'Control', 'default': 'Default friction', 'reintervention': 'Reintervention'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

panels = [
    (pivot_open,    ['control', 'default', 'reintervention'], 'P(open) — 7-day rolling avg',
     'A. Opening Rate (Individual Trajectories + Conditional means)', 'Thin lines = participants; thick lines = condition means', 4),
    (pivot_dismiss, ['control', 'default', 'reintervention'], 'P(dismiss) — 7-day rolling avg',
     'B. Dismissal Rate (Individual Trajectories + Conditional means)', 'Thin lines = participants; thick lines = condition means', 2),
]

for ax, (pivot, conds, ylabel, title, note, label_pos) in zip(axes, panels):
    days_x = pivot.columns.to_numpy(dtype=float).astype(int)  # numpy int array for matplotlib

    # Thin individual lines
    for pid in pivot.index:
        cond = pid_condition.get(pid)
        if cond not in conds:
            continue
        ax.plot(days_x, pivot.loc[pid].to_numpy(dtype=float),
                color=COLORS[cond], alpha=0.3, linewidth=0.6, zorder=1)

    # Thick group mean lines
    for cond in conds:
        pids_cond = [p for p in pivot.index if pid_condition.get(p) == cond]
        if not pids_cond:
            continue
        group_mean = pivot.loc[pids_cond].mean(skipna=True).to_numpy(dtype=float)
        ax.plot(days_x, group_mean,
                color=COLORS[cond], linewidth=2.8, label=LABELS[cond], zorder=5)

    ax.set_xlabel('Days since first event', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.legend(fontsize=10, loc=label_pos)
    ax.set_xlim(0, 42)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25, linestyle='--')
    if note:
        ax.text(0.02, 0.02, note, transform=ax.transAxes,
                fontsize=8, color='#757575', style='italic')

fig.suptitle(
    'Individual Trajectories vs Conditonal Group Means\n'
    '(7-day rolling average; participants with ≥ 14 active days shown)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
COLORS = {'control': "#218D0E", 'default': '#1976D2', 'reintervention': '#F57C00'}
LABELS = {'control': 'Control', 'default': 'Default friction', 'reintervention': 'Reintervention'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

panels = [
    (pivot_open,    ['control', 'default', 'reintervention'], 'P(open) — 7-day rolling avg',
     'A. Opening Rate (Conditional means)', 'thick lines = condition means', 4, False),
    (pivot_open,    ['control', 'default', 'reintervention'], 'P(open) — 7-day rolling avg',
     'B. Opening Rate (Individual Trajectories + Conditional means)', 'Thin lines = participants; thick lines = condition means', 4, True),
]

for ax, (pivot, conds, ylabel, title, note, label_pos, plot_version) in zip(axes, panels):
    days_x = pivot.columns.to_numpy(dtype=float).astype(int)  # numpy int array for matplotlib

    # Thin individual lines
    if plot_version == True:
        for pid in pivot.index:
            cond = pid_condition.get(pid)
            if cond not in conds:
                continue
            ax.plot(days_x, pivot.loc[pid].to_numpy(dtype=float),
                    color=COLORS[cond], alpha=0.3, linewidth=0.6, zorder=1)

    # Thick group mean lines
    for cond in conds:
        pids_cond = [p for p in pivot.index if pid_condition.get(p) == cond]
        if not pids_cond:
            continue
        group_mean = pivot.loc[pids_cond].mean(skipna=True).to_numpy(dtype=float)
        ax.plot(days_x, group_mean,
                color=COLORS[cond], linewidth=2.8, label=LABELS[cond], zorder=5)

    ax.set_xlabel('Days since first event', fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.set_title(title, fontsize=15, fontweight='bold', pad=10)
    ax.legend(fontsize=15, loc=label_pos)
    ax.set_xlim(0, 42)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25, linestyle='--')
    if note:
        ax.text(0.02, 0.02, note, transform=ax.transAxes,
                fontsize=12, color='#757575', style='italic')

#fig.suptitle(
#    'Individual Trajectories vs Conditonal Group Means\n'
#    '(7-day rolling average; participants with ≥ 14 active days shown)',
#    fontsize=13, fontweight='bold'
#)
plt.tight_layout()

plt.savefig("../outputs/playground_outputs/ds5_indivTrajectory_groupMean.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Map study condition onto behavioral data
# ------------------------------------------------------------

condition_map = (
    df_feat
    .set_index('participant_id_canonical')['study_condition']
    .to_dict()
)

df_beh['study_condition'] = df_beh['participant_id_canonical'].map(condition_map)

# study_condition is used as a groupby key below, and pandas drops NaN keys silently.
# Expected: exactly 1 participant ('test-<participant-id>', a test
# account with no pre-survey record — see memory ds5_structural_discoveries.md §3), so
# the effective N for every figure below is 572, not the 573 loaded in cell 3.
_unmapped = sorted(df_beh.loc[df_beh['study_condition'].isna(), 'participant_id_canonical'].unique())
print(f'Participants without study_condition (dropped by later groupbys): {len(_unmapped)} -> {_unmapped}')
assert len(_unmapped) <= 1, 'Unexpected number of participants without a study_condition'


# ------------------------------------------------------------
# 2. Parse timestamp and create time variables
# ------------------------------------------------------------

df_beh['event_timestamp_local'] = pd.to_datetime(
    df_beh['event_timestamp_local'],
    errors='coerce'
)

df_beh['event_hour'] = df_beh['event_timestamp_local'].dt.hour

df_beh['day_int'] = np.floor(
    pd.to_numeric(df_beh['days_since_first_event'], errors='coerce')
)

n_nan = df_beh['day_int'].isna().sum()
if n_nan > 0:
    print(f'Note: {n_nan} rows have NaN day_int and will be excluded from daily aggregations')


# ------------------------------------------------------------
# 3. Basic summary
# ------------------------------------------------------------

cond_counts = df_beh.groupby('study_condition')['participant_id_canonical'].nunique()

print('Participants with behavioral data per condition:')
print(cond_counts)

valid_days = df_beh['day_int'].dropna()

print(
    f'\nObservation window: day {int(valid_days.min())} '
    f'to day {int(valid_days.max())}'
)


# ------------------------------------------------------------
# 4. Per-user per-day aggregation
# ------------------------------------------------------------

daily = (
    df_beh
    .groupby(['participant_id_canonical', 'day_int', 'study_condition'])
    .agg(
        n_events    = ('event_outcome_simple', 'count'),
        n_opened    = ('event_outcome_simple', lambda x: (x == 'opened').sum()),
        n_dismissed = ('event_outcome_simple', lambda x: (x == 'dismissed').sum()),
        n_closed    = ('event_outcome_simple', lambda x: (x == 'closed').sum()),
    )
    .reset_index()
)

daily['open_rate'] = daily['n_opened'] / daily['n_events']
daily['dismiss_rate'] = daily['n_dismissed'] / daily['n_events']


# ------------------------------------------------------------
# 5. Filter to participants with at least 14 active days
# ------------------------------------------------------------

active_days = daily.groupby('participant_id_canonical')['day_int'].count()

eligible = set(active_days[active_days >= 14].index)

print(
    f'Participants with >= 14 active days: '
    f'{len(eligible)} / {daily["participant_id_canonical"].nunique()}'
)

daily_elig = daily[daily['participant_id_canonical'].isin(eligible)].copy()

pid_condition = (
    daily_elig
    .drop_duplicates('participant_id_canonical')
    .set_index('participant_id_canonical')['study_condition']
    .to_dict()
)


# ------------------------------------------------------------
# 6. Build participant x day pivot tables and smooth them
# ------------------------------------------------------------

def make_pivot_smooth(daily_df, value_col, window=7):
    """
    Creates a participant x day matrix and applies a rolling average over days.

    Rows:
        participants

    Columns:
        days since first event

    Values:
        selected behavioral rate, for example open_rate or dismiss_rate
    """

    pivot = daily_df.pivot_table(
        index='participant_id_canonical',
        columns='day_int',
        values=value_col
    )

    # Rolling across days.
    # Transpose so days are rows, apply rolling mean, then transpose back.
    pivot_smooth = pivot.T.rolling(
        window=window,
        min_periods=1
    ).mean().T

    return pivot_smooth


pivot_open = make_pivot_smooth(daily_elig, 'open_rate', window=7)
pivot_dismiss = make_pivot_smooth(daily_elig, 'dismiss_rate', window=7)

print(f'Pivot shape: {pivot_open.shape}  (participants x days)')


# ------------------------------------------------------------
# 7. Plot settings
# ------------------------------------------------------------

COLORS = {
    'control': '#218D0E',
    'default': '#1976D2',
    'reintervention': '#F57C00'
}

LABELS = {
    'control': 'Control',
    'default': 'Default friction',
    'reintervention': 'Reintervention'
}


# ------------------------------------------------------------
# 8. Helper functions for plotting
# ------------------------------------------------------------

def get_days_x(pivot):
    return pivot.columns.to_numpy(dtype=float).astype(int)


def get_pids_for_condition(pivot, condition):
    return [
        pid for pid in pivot.index
        if pid_condition.get(pid) == condition
    ]


def condition_mean(pivot, condition):
    pids_cond = get_pids_for_condition(pivot, condition)

    if not pids_cond:
        return None

    return pivot.loc[pids_cond].mean(skipna=True)


def plot_individuals_and_means(
    ax,
    pivot,
    conditions,
    ylabel,
    title,
    note=None,
    show_legend=True
):
    """
    Plots thin individual trajectories and thick condition means.
    """

    days_x = get_days_x(pivot)

    # Thin individual lines
    for pid in pivot.index:
        cond = pid_condition.get(pid)

        if cond not in conditions:
            continue

        ax.plot(
            days_x,
            pivot.loc[pid].to_numpy(dtype=float),
            color=COLORS[cond],
            alpha=0.22,
            linewidth=0.55,
            zorder=1
        )

    # Thick condition mean lines
    for cond in conditions:
        mean_series = condition_mean(pivot, cond)

        if mean_series is None:
            continue

        ax.plot(
            days_x,
            mean_series.to_numpy(dtype=float),
            color=COLORS[cond],
            linewidth=3.2,
            label=LABELS[cond],
            zorder=5
        )

    ax.set_xlabel('Days since first event', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    # Only 15/378 participants have any raw observation at day 42 (242 end at day 41);
    # the 7-day rolling window would otherwise carry stale values past study end.
    ax.set_xlim(0, 41)
    ax.set_ylim(-0.05, 1.05)

    ax.grid(alpha=0.25, linestyle='--')

    if show_legend:
        ax.legend(fontsize=9, loc='best')

    if note:
        ax.text(
            0.02,
            0.02,
            note,
            transform=ax.transAxes,
            fontsize=8,
            color='#757575',
            style='italic'
        )


def plot_treatment_contrast_opening(ax, pivot_open):
    """
    Plots the average treatment contrast over time.

    This is not a full causal model.
    It is a descriptive difference in condition means:

        Default friction - Control
        Reintervention - Control

    Negative values mean that the intervention condition has a lower
    opening probability than control.
    """

    days_x = get_days_x(pivot_open)

    control_mean = condition_mean(pivot_open, 'control')
    default_mean = condition_mean(pivot_open, 'default')
    reint_mean = condition_mean(pivot_open, 'reintervention')

    if control_mean is None:
        raise ValueError('No control participants found in pivot_open.')

    # Horizontal reference line at zero effect
    ax.axhline(
        0,
        color='black',
        linewidth=1.0,
        linestyle='--',
        alpha=0.7,
        zorder=1
    )

    if default_mean is not None:
        effect_default = default_mean - control_mean

        ax.plot(
            days_x,
            effect_default.to_numpy(dtype=float),
            color=COLORS['default'],
            linewidth=3.0,
            label='Default friction − Control',
            zorder=5
        )

    if reint_mean is not None:
        effect_reint = reint_mean - control_mean

        ax.plot(
            days_x,
            effect_reint.to_numpy(dtype=float),
            color=COLORS['reintervention'],
            linewidth=3.0,
            label='Reintervention − Control',
            zorder=5
        )

    ax.set_xlabel('Days since first event', fontsize=11)
    ax.set_ylabel('Difference in P(open)', fontsize=11)

    ax.set_title(
        'Average Treatment Contrast Over Time\n'
        'Opening rate: intervention mean − control mean',
        fontsize=12,
        fontweight='bold',
        pad=10
    )

    ax.set_xlim(0, 41)

    # You can adjust these limits if your effects are larger or smaller.
    ax.set_ylim(-0.5, 0.2)

    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(fontsize=9, loc='best')

    ax.text(
        0.02,
        0.02,
        ('Negative values = lower opening probability than control. '
         'Contributing n per arm falls ~123-128 (day 0) to ~98-101 (day 41): '
         'late-window bends may be composition change, not a within-person trend.'),
        transform=ax.transAxes,
        fontsize=8,
        color='#757575',
        style='italic'
    )


# ------------------------------------------------------------
# 9. Create the final three-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(22, 6))


# Panel 1: Opening rate
plot_individuals_and_means(
    ax=axes[0],
    pivot=pivot_open,
    conditions=['control', 'default', 'reintervention'],
    ylabel='P(open) — 7-day rolling average',
    title='A. Opening Rate\nIndividual trajectories + condition means',
    note=('Thin lines = participants; thick lines = condition means\n'
          'Contributing n per arm (7-day smoothed) falls 123/128/127 at day 0 '
          'to 98/100/101 at day 41 — the late tail is a partly self-selected subsample')
)


# Panel 2: Dismissal rate
plot_individuals_and_means(
    ax=axes[1],
    pivot=pivot_dismiss,
    conditions=['control','default', 'reintervention'],
    ylabel='P(dismiss) — 7-day rolling average',
    title='B. Dismissal Rate\nAll three conditions',
    note='Control is shown: control saw no breathing delay but could still dismiss (mean rate_dismissed = 0.179)'
)


# Panel 3: Treatment contrast over time
plot_treatment_contrast_opening(
    ax=axes[2],
    pivot_open=pivot_open
)


fig.suptitle(
    'Individual Trajectories, Condition Means, and Average Treatment Contrast\n'
    '(7-day rolling average; participants with ≥ 14 active days shown)',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

---
## Version B — Same Average, Different Dynamics

**Message:** Average opening rate (x_mean) is *similar* for the three users shown here
(0.52–0.61), but their behavioral dynamics differ sharply — in event volume, temporal
spread, and day-to-day variability.

**User types (as actually selected by the scoring code below; H_hour maximum = log2(24) = 4.585):**
- **User A** — High volume, temporally diffuse: 190.1 events/day, H_hour=4.04 (88% of maximum), low day-to-day noise (sig_hat=0.174)
- **User B** — Low volume, calmer: 2.1 events/day, H_hour=3.08 (67% of maximum — a broad handful of hours, not 1–2 peak hours), sig_hat=0.482
- **User C** — Low volume, most volatile: 2.7 events/day, H_hour=3.88, highest day-to-day noise (sig_hat=0.684)

B and C have nearly the same event volume; what actually separates them is sig_hat.

**Note on session duration:** DS5 only captures OneSec intervention events, not session durations.
The "concentrated" pattern is approximated by low events_per_day + low hour_entropy.

In [ ]:
# Merge feature sources needed for user selection
feat_b = (
    df_feat[['participant_id_canonical', 'events_per_day', 'hour_entropy']]
    .merge(
        df_pheno_feat[['participant_id_canonical', 'sig_hat', 'x_mean']],
        on='participant_id_canonical', how='inner'
    )
)

# Target: similar x_mean in [0.50, 0.80]
target = (
    feat_b[feat_b['x_mean'].between(0.50, 0.80)]
    .dropna(subset=['events_per_day', 'hour_entropy', 'sig_hat'])
    .copy()
)

# Normalize columns for scoring
for col in ['events_per_day', 'hour_entropy', 'sig_hat']:
    target[f'{col}_z'] = (target[col] - target[col].mean()) / target[col].std()

# Score each user type
target['score_A'] =  target['events_per_day_z'] + target['hour_entropy_z']   # impulsive
target['score_B'] = -target['events_per_day_z'] - target['hour_entropy_z']   # concentrated
target['score_C'] =  target['sig_hat_z']                                       # volatile

# Select top candidate per type, no overlaps
used = set()
selected = {}
for user_type, score_col in [('A', 'score_A'), ('B', 'score_B'), ('C', 'score_C')]:
    candidates = target[~target['participant_id_canonical'].isin(used)].nlargest(1, score_col)
    if len(candidates) > 0:
        row = candidates.iloc[0]
        selected[user_type] = row['participant_id_canonical']
        used.add(row['participant_id_canonical'])

print('Selected users:')
for ut, pid in selected.items():
    row = target[target['participant_id_canonical'] == pid].iloc[0]
    print(f'  User {ut}: x_mean={row["x_mean"]:.3f}  sig_hat={row["sig_hat"]:.3f}  '
          f'events/day={row["events_per_day"]:.1f}  H_hour={row["hour_entropy"]:.3f}')

In [ ]:
USER_COLORS = {'A': '#E53935', 'B': '#1E88E5', 'C': '#43A047'}
USER_LABELS = {
    'A': 'User A — High volume\n(190 events/day, spread across all hours)',
    'B': 'User B — Low volume, calmer\n(2.1 events/day, sigma=0.48)',
    'C': 'User C — Low volume, volatile\n(2.7 events/day, sigma=0.68)',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col_idx, (user_type, pid) in enumerate(selected.items()):
    user_beh = df_beh[df_beh['participant_id_canonical'] == pid].dropna(subset=['day_int'])
    row = target[target['participant_id_canonical'] == pid].iloc[0]
    color = USER_COLORS[user_type]

    # --- Row 0: Hour-of-day event distribution ---
    ax_h = axes[0, col_idx]
    hourly = user_beh.groupby('event_hour').size().reindex(range(24), fill_value=0)
    ax_h.bar(np.arange(24), hourly.values,
             color=color, alpha=0.82, edgecolor='white', linewidth=0.5)
    ax_h.set_xlabel('Hour of day', fontsize=10)
    ax_h.set_ylabel('Events', fontsize=10)
    ax_h.set_title(
        f'{USER_LABELS[user_type]}\n'
        f'x_mean={row["x_mean"]:.2f}  |  σ={row["sig_hat"]:.3f}\n'
        f'events/day={row["events_per_day"]:.1f}  |  H_hour={row["hour_entropy"]:.2f}',
        fontsize=9, fontweight='bold'
    )
    ax_h.set_xlim(-0.5, 23.5)
    ax_h.set_xticks([0, 6, 12, 18, 23])
    ax_h.grid(axis='y', alpha=0.3)

    # --- Row 1: Daily event count over time ---
    ax_d = axes[1, col_idx]
    user_daily = user_beh.groupby('day_int').size().reset_index(name='n')
    days_arr = user_daily['day_int'].to_numpy(dtype=int)
    n_arr    = user_daily['n'].to_numpy(dtype=float)
    smooth   = user_daily['n'].rolling(7, min_periods=1).mean().to_numpy(dtype=float)

    ax_d.bar(days_arr, n_arr, color=color, alpha=0.5, width=1.0)
    ax_d.plot(days_arr, smooth, color=color, linewidth=2.0, label='7-day avg')
    ax_d.set_xlabel('Days since first event', fontsize=10)
    ax_d.set_ylabel('Events per day', fontsize=10)
    ax_d.set_title(f'Daily event count — User {user_type}', fontsize=10)
    ax_d.legend(fontsize=8)
    ax_d.grid(alpha=0.25, linestyle='--')

x_means = [target[target['participant_id_canonical'] == pid].iloc[0]['x_mean']
           for pid in selected.values()]
fig.suptitle(
    f'Version B — Similar Average Opening Rate '
    f'({np.min(x_means):.2f}–{np.max(x_means):.2f}; mean {np.mean(x_means):.2f}), '
    'Different Behavioral Dynamics\n'
    '(Three real DS5 participants selected to illustrate distinct behavioral signatures)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Version C — Intervention Response Phenotypes

**Message:** Participants in the intervention conditions show substantially different response trajectories.
Grouping by response type (rather than condition alone) reveals distinct profiles that connect naturally
to the behavioral phenotypes from Stage 4 Step 2.

**Response type classification (intervention groups only):**
- **Strong responder:** opening rate dropped > 10pp from early to late third
- **Weak responder:** dropped 2–10pp
- **Non-responder:** stable (|change| ≤ 2pp)
- **Increase:** opening rate increased (> 2pp) — could reflect adaptation back to baseline or reactance

⚠ **Caveat:** With only two time windows (early / late calendar thirds), we cannot distinguish
'adapters' (initial drop then return) from 'reactance' (sustained increase). A continuous
daily trajectory would be needed to distinguish these.

In [ ]:
# Intervention participants only, with adaptation features
df_int = (
    df_feat[df_feat['study_condition'].isin(['default', 'reintervention'])]
    .dropna(subset=['open_rate_early_third_calendar', 'open_rate_late_third_calendar',
                    'open_rate_change_calendar'])
    .copy()
)

def classify_response(change):
    """open_rate_change_calendar = late_third - early_third (negative = improvement)."""
    if change < -0.10:
        return 'Strong responder'
    elif change < -0.02:
        return 'Weak responder'
    elif change <= 0.02:
        return 'Non-responder'
    else:
        return 'Increase (adapt/reactance)'

df_int['response_type'] = df_int['open_rate_change_calendar'].apply(classify_response)

print('Response type distribution (intervention participants):')
print(df_int['response_type'].value_counts())
print(f'\nTotal: {len(df_int)} intervention participants with adaptation features')

In [ ]:
RESP_COLORS = {
    'Strong responder':           '#2E7D32',
    'Weak responder':             '#81C784',
    'Non-responder':              '#78909C',
    'Increase (adapt/reactance)': '#E53935',
}
RESP_ORDER = ['Strong responder', 'Weak responder', 'Non-responder', 'Increase (adapt/reactance)']

resp_map = df_int.set_index('participant_id_canonical')['response_type'].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: mean trajectory per response type ---
ax = axes[0]
pivot_int = pivot_open[pivot_open.index.isin(df_int['participant_id_canonical'].tolist())]
days_x_c  = pivot_int.columns.to_numpy(dtype=float).astype(int)

for resp_type in RESP_ORDER:
    type_pids = [p for p, r in resp_map.items() if r == resp_type and p in pivot_int.index]
    if not type_pids:
        continue
    n = len(type_pids)
    for pid in type_pids:
        ax.plot(days_x_c, pivot_int.loc[pid].to_numpy(dtype=float),
                color=RESP_COLORS[resp_type], alpha=0.08, linewidth=0.6)
    group_mean = pivot_int.loc[type_pids].mean(skipna=True).to_numpy(dtype=float)
    ax.plot(days_x_c, group_mean,
            color=RESP_COLORS[resp_type], linewidth=2.5,
            label=f'{resp_type} (N={n})')

ax.set_xlabel('Days since first event', fontsize=11)
ax.set_ylabel('P(open) — 7-day rolling avg', fontsize=11)
ax.set_title('Mean trajectory by intervention response type\n(intervention conditions only; ≥ 14 active days)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 42)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.25, linestyle='--')
ax.text(0.02, 0.02,
        '⚠ Cannot distinguish adaptation from reactance\nwith only early/late window comparison',
        transform=ax.transAxes, fontsize=8, color='#757575', style='italic')

# --- Right: response type by behavioral phenotype ---
ax2 = axes[1]
resp_pheno = (
    df_int[['participant_id_canonical', 'response_type']]
    .merge(df_clusters[['participant_id_canonical', 'cluster_behav']],
           on='participant_id_canonical', how='left')
)
resp_pheno_valid = resp_pheno.dropna(subset=['cluster_behav'])
n_matched = len(resp_pheno_valid)
n_total   = len(resp_pheno)

cross = pd.crosstab(resp_pheno_valid['cluster_behav'], resp_pheno_valid['response_type'],
                    normalize='index')
plot_resp = [r for r in RESP_ORDER if r in cross.columns]
cross = cross[plot_resp].reindex([c for c in ['C1', 'C2', 'C3'] if c in cross.index])

bottom = np.zeros(len(cross))
x_pos  = np.arange(len(cross))
for resp_type in plot_resp:
    vals = cross[resp_type].values
    ax2.bar(x_pos, vals, bottom=bottom, color=RESP_COLORS[resp_type],
            label=resp_type, edgecolor='white', linewidth=0.8)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.07:
            ax2.text(xi, b + v / 2, f'{v:.0%}', ha='center', va='center',
                     fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax2.set_xticks(x_pos)
ax2.set_xticklabels(cross.index.tolist(), fontsize=10)
ax2.set_xlabel('Behavioral phenotype (Path A cluster)', fontsize=11)
ax2.set_ylabel('Proportion', fontsize=11)
ax2.set_title(f'Response type by phenotype cluster\n({n_matched}/{n_total} with cluster label)',
              fontsize=11, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.grid(axis='y', alpha=0.3)

fig.suptitle('Version C — Intervention Response Phenotypes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Version D — Decision-Level Intervention Flow

**Message:** Different users have different probabilities at each step of the friction decision:
attempt → delay shown → open / dismiss / close.

**Left panel:** Outcome distribution by condition — shows how friction shifts the aggregate outcome mix.  
**Right panel:** Per-participant scatter (open vs dismiss rate on friction events) — shows heterogeneity
within each condition, marked by behavioral phenotype cluster.

⚠ **Control group:** No friction events shown → excluded from intervention-event scatter (right panel).
Their rate_opened in the left panel (0.763 on average) reflects opening without a breathing delay — control users could still dismiss (0.179) or close (0.058).

In [ ]:
# Version D inputs — RESTORED 2026-08-08. The original cell that built these was deleted
# after it ran, leaving cell 18 referencing undefined names (NameError on a fresh run).
# Left panel: participant-averaged outcome rates per condition. Verified to reproduce the
# stored figure exactly (control 76.3/17.9/5.8, default 52.9/39.9/7.2, reint 61.0/31.0/8.0).
cond_flow = (
    df_feat.dropna(subset=['study_condition'])
    .groupby('study_condition')[['rate_opened', 'rate_dismissed', 'rate_closed']]
    .mean()
)

# Right panel: per-participant open-vs-dismiss split over decision events only
# (closed events are excluded from the denominator, so int_opened + int_dismissed == 1).
_ev = df_beh[df_beh['event_outcome_simple'].isin(['opened', 'dismissed'])]
_counts = (
    _ev.groupby('participant_id_canonical')['event_outcome_simple']
    .value_counts().unstack(fill_value=0)
)
_tot = _counts.sum(axis=1)
df_flow = pd.DataFrame({
    'int_opened':    _counts.get('opened', 0) / _tot,
    'int_dismissed': _counts.get('dismissed', 0) / _tot,
}).reset_index()
df_flow['study_condition'] = df_flow['participant_id_canonical'].map(condition_map)
df_flow = df_flow.merge(
    df_clusters[['participant_id_canonical', 'cluster_behav']],
    on='participant_id_canonical', how='left',
)
print((cond_flow * 100).round(1))
print(f'df_flow rows: {len(df_flow)}  with cluster label: {df_flow["cluster_behav"].notna().sum()}')


In [ ]:
COND_ORDER  = ['control', 'default', 'reintervention']
COND_LABELS = {'control': 'Control', 'default': 'Default\nfriction', 'reintervention': 'Reintervention'}
FLOW_COLS   = ['rate_opened', 'rate_dismissed', 'rate_closed']
FLOW_LABELS = ['Opened', 'Dismissed', 'Closed / other']
FLOW_COLORS = ['#EF5350', '#42A5F5', '#66BB6A']

COND_SCATTER_COLORS = {'control': '#9E9E9E', 'default': '#1976D2', 'reintervention': '#F57C00'}
CLUSTER_MARKERS     = {'C1': 'o', 'C2': 's', 'C3': '^'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: stacked bar by condition ---
ax = axes[0]
x      = np.arange(len(COND_ORDER))
bottom = np.zeros(len(COND_ORDER))

for col, label, color in zip(FLOW_COLS, FLOW_LABELS, FLOW_COLORS):
    vals = [cond_flow.loc[c, col] if c in cond_flow.index else 0.0 for c in COND_ORDER]
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor='white', linewidth=0.9)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(xi, b + v / 2, f'{v:.1%}', ha='center', va='center',
                    fontsize=10, color='white', fontweight='bold')
    bottom += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels([COND_LABELS[c] for c in COND_ORDER], fontsize=11)
ax.set_ylabel('Proportion of events', fontsize=11)
ax.set_title('Outcome distribution by condition\n(all events)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
ax.text(0.02, 0.02,
        'Control: no breathing delay, but users could still dismiss or close (mean rate_opened = 0.763)',
        transform=ax.transAxes, fontsize=8, color='#757575', style='italic')

# --- Right: per-participant scatter (intervention events only) ---
ax2 = axes[1]
# Only plot if int_opened and int_dismissed columns exist
if 'int_opened' in df_flow.columns and 'int_dismissed' in df_flow.columns:
    scatter_df = df_flow.dropna(subset=['int_opened', 'int_dismissed'])
    for cond in ['default', 'reintervention']:
        sub_cond = scatter_df[scatter_df['study_condition'] == cond]
        for cluster, marker in CLUSTER_MARKERS.items():
            sub = sub_cond[sub_cond['cluster_behav'] == cluster]
            if len(sub) == 0:
                continue
            ax2.scatter(
                sub['int_opened'].values,
                sub['int_dismissed'].values,
                c=COND_SCATTER_COLORS[cond], marker=marker,
                alpha=0.55, s=35, edgecolors='none'
            )
else:
    ax2.text(0.5, 0.5, 'No intervention events found', ha='center', va='center',
             transform=ax2.transAxes)

# Legend handles
cond_patches = [
    mpatches.Patch(color=COND_SCATTER_COLORS[c], label=COND_LABELS[c])
    for c in ['default', 'reintervention']
]
cluster_handles = [
    plt.Line2D([0], [0], marker=m, color='#444', linestyle='None', markersize=8, label=f'Cluster {cl}')
    for cl, m in CLUSTER_MARKERS.items()
]
ax2.legend(handles=cond_patches + cluster_handles, fontsize=9, loc='upper right')

ax2.set_xlabel('P(opened | friction event)', fontsize=11)
ax2.set_ylabel('P(dismissed | friction event)', fontsize=11)
ax2.set_title('Per-participant flow (friction events only)\ncolor = condition, shape = phenotype cluster',
              fontsize=12, fontweight='bold')
ax2.set_xlim(-0.05, 1.05)
ax2.set_ylim(-0.05, 1.05)
ax2.plot([0, 1], [1, 0], 'k--', alpha=0.2, linewidth=1)
ax2.grid(alpha=0.25, linestyle='--')
ax2.text(0.02, 0.02,
         '⚠ Control group excluded (no friction events).\nParticipants without cluster label omitted.',
         transform=ax2.transAxes, fontsize=8, color='#757575', style='italic')

fig.suptitle('Version D — Decision-Level Intervention Flow', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary — Feasibility Assessment

| Version | Verdict | Key finding |
|---|---|---|
| A — Individual Trajectories | ✅ High | Individual lines show substantial spread around group means; motivates phenotyping |
| B — Same Average, Different Dynamics | ✅ High | Three real users illustrate genuinely distinct temporal profiles with nearly identical x_mean |
| C — Response Phenotypes | ✅ Medium-High | Response types are visible; adapter vs reactance distinction requires a pre-intervention baseline |
| D — Decision Flow | ✅ High | Friction conditions shift outcome mix; per-participant scatter reveals within-condition heterogeneity |

**Strongest empirical motivation figure for presentation:** Version A (individual trajectories) or
Version C (response phenotypes) — both directly support the argument that average effects mask
behavioral heterogeneity.

**Version B** works best as a conceptual bridge figure between screen time research and CAS framing.

**Version D** works best as a mechanistic figure connecting the intervention mechanism to the phenotype structure.